In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Se crea una sesión de spark, y que se guarde en mongo
# 1. Cargar datos
spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1") \
    .getOrCreate()

df = spark.read.format("mongodb").load()

In [6]:
#Verificar que el spark guarda sus datos (en la ram)
df.printSchema()
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- pagina: integer (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+--------------+--------------------+------+-----+
|                 _id|categoria|      fecha_captura|         grupo|                item|pagina|valor|
+--------------------+---------+-------------------+--------------+--------------------+------+-----+
|69dd827d02f3469f0...|   Travel|2026-04-13 23:55:41|      G1_Libro|  It's Only Himalaya|  NULL|45.17|
|69dd827d02f3469f0...|   Travel|2026-04-13 23:55:41|      G1_Libro|Full Moon over No...|  NULL|49.43|
|69dd82922caa11bee...|  Science|2026-04-13 23:56:01|MercadoLaboral|The Most Perfect ...|     1|42.96|
|69dd82922caa11bee...|  Science|2026-04-13 23:56:01|MercadoLaboral|Immunity: How Eli...|     1|57.36|
|69dd82922

In [7]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 16
Registros validos: 16


In [8]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|  It's Only Himalaya|45.17|
|Full Moon over No...|49.43|
|The Most Perfect ...|42.96|
|Immunity: How Eli...|57.36|
|Sorting the Beef ...|44.74|
|Tipping Point for...|37.55|
|The Fabric of the...|55.91|
|Diary of a Citize...|28.41|
|The Origin of Spe...|10.01|
|    The Grand Design|13.76|
|Peak: Secrets fro...|16.28|
|The Elegant Unive...|13.03|
|The Disappearing ...|57.35|
|Surely You're Jok...|25.83|
|Seven Brief Lesso...| 30.6|
|    The Selfish Gene|29.45|
+--------------------+-----+



In [9]:
df.filter(df["valor"] > 50).show()

+--------------------+---------+-------------------+--------------+--------------------+------+-----+
|                 _id|categoria|      fecha_captura|         grupo|                item|pagina|valor|
+--------------------+---------+-------------------+--------------+--------------------+------+-----+
|69dd82922caa11bee...|  Science|2026-04-13 23:56:01|MercadoLaboral|Immunity: How Eli...|     1|57.36|
|69dd82922caa11bee...|  Science|2026-04-13 23:56:01|MercadoLaboral|The Fabric of the...|     1|55.91|
|69dd82922caa11bee...|  Science|2026-04-13 23:56:01|MercadoLaboral|The Disappearing ...|     1|57.35|
+--------------------+---------+-------------------+--------------+--------------------+------+-----+



In [10]:
df.groupBy("grupo").count().show()

+--------------+-----+
|         grupo|count|
+--------------+-----+
|      G1_Libro|    2|
|MercadoLaboral|   14|
+--------------+-----+



In [11]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+-----------------+-------------+-------------+
|categoria|cantidad_libros|  precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+-----------------+-------------+-------------+
|   Travel|              2|             47.3|        45.17|        49.43|
|  Science|             14|33.08857142857143|        10.01|        57.36|
+---------+---------------+-----------------+-------------+-------------+



In [12]:
fila = df.orderBy(F.desc("valor")).select(
    "item", "categoria", "fecha_captura", "valor"
).first()

print("Producto más caro:", fila["item"])
print("Categoría:", fila["categoria"])
print("Fecha y hora de captura:", fila["fecha_captura"])
print("Valor:", fila["valor"])

Producto más caro: Immunity: How Elie Metchnikoff Changed the Course of Modern Medicine
Categoría: Science
Fecha y hora de captura: 2026-04-13 23:56:01
Valor: 57.36


In [13]:
from pyspark.sql import functions as F

NOMBRE_GRUPO = "MercadoLaboral"

ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
     .groupBy("categoria") \
     .agg(
          F.count("item").alias("total_libros"),
          F.round(F.avg("valor"), 2).alias("precio_medio"),
          F.max("fecha_captura").alias("ultima_sincronizacion")
     )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()


--- TICKET DE SALIDA: MercadoLaboral ---
+---------+------------+------------+---------------------+
|categoria|total_libros|precio_medio|ultima_sincronizacion|
+---------+------------+------------+---------------------+
|  Science|          14|       33.09|  2026-04-13 23:56:02|
+---------+------------+------------+---------------------+

